# Ask CIP v8b: the fine-tuned search model, tested fairly

v8 fine-tuned **bge-small** on our practice questions. On Colab it ranks the right survey question first far more often
(golden 79.5% → 86-91%, fresh 59.6% → 75-79%, in the first 14 for 100% of golden), at today's size and speed. Inside Ask
on the Mac it scored golden 152 against today's 154, but that copy was **8-bit** (its vectors 0.966-0.969 of PyTorch's)
while today's file is 16-bit, so part of the gain may have been lost to the copy. This notebook repeats v8's fine-tune and
exports at full precision (32-bit: PyTorch's own vectors), so the test inside Ask compares the models, not the copies:

1. today's bge-small, plain and with its search instruction, scored as Ask ranks (the yardstick, as in v8);
2. both fine-tuned as in v8 (practice questions and one analyst's ask per survey question, a hard negative each, one epoch);
3. both exported as Ask loads them (32-bit ONNX, tokenizer, calibration.json), the vectors checked against PyTorch's.

**Run:** Runtime → Change runtime type → **L4** (or A100) → Runtime → Run all → pick `v8-search.json` from
`Downloads/cip-colab`. About 15-20 minutes (~2-4 units). `ask-v8b.zip` (about 250 MB) downloads at the end: send it
back, it is scored on the Mac inside Ask. A model that wins there gets a 16-bit copy (as today's 66 MB file), checked
again, before it ships. No API key is used; the "HF_TOKEN" warning is harmless.


In [ ]:
# 1. Libraries, the GPU, v8-search.json (Downloads/cip-colab), and the results kept as we go
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "sentence-transformers", "datasets", "onnx", "onnxruntime", "einops"], check=True)
import gc, json, os, random, shutil, time
import numpy as np, torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → L4, then Run all again"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
NEEDED = ("v8-search.json",)
if not all(os.path.exists(f) for f in NEEDED):
    from google.colab import files
    print("Pick v8-search.json from Downloads/cip-colab")
    files.upload()
search = json.load(open("v8-search.json"))
os.makedirs("ask-v8b", exist_ok=True)
results = {"embedders": {}}

def save():
    json.dump(results, open("ask-v8b/results.json", "w"), indent=1)

def free():
    gc.collect(); torch.cuda.empty_cache()

print(len(search["docs"]), "survey questions ·", len(search["practice"]), "practice questions ·",
      {t: len(v) for t, v in search["tests"].items()})


In [ ]:
# 2. The search score, as checked on the Mac with today's model (golden: right one first 57.6% by vectors alone, 98% in
# the first 14 with the asks and standing, as inside Ask): each question's rank of its first right survey question
docs = search["docs"]
at = {d["key"]: i for i, d in enumerate(docs)}
owners = [(i, a) for i, d in enumerate(docs) for a in d["asks"]]
ask_owner = np.array([i for i, _ in owners])
standing = np.array([d["standing"] for d in docs], np.float32)
# settings are chosen on 2,000 practice questions that are not word for word an embedded ask (57% are: they would
# match themselves), never on the tests
said = {a.strip().lower() for d in docs for a in d["asks"]}
tune = [p for p in search["practice"] if p["q"].strip().lower() not in said]
random.Random(8).shuffle(tune)
tune = tune[:2000]
tune_q = {p["q"] for p in tune}
SETS = {"tune": [{"subject": p["subject"], "right": [p["key"]]} for p in tune],
        **{t: [r for r in rows if r["right"]] for t, rows in search["tests"].items()}}
RIGHT = {t: [{at[k] for k in r["right"] if k in at} for r in rows] for t, rows in SETS.items()}

def fused(q, doc_vecs, ask_vecs, w, stand):
    """Closeness as Ask ranks it (api/ask_index.py retrieve): the survey question's own, joined with its nearest ask's
    (weight w), plus its standing (long-running and recent)."""
    near = q @ doc_vecs.T
    if w:
        nearest_ask = near.copy()
        sims = q @ ask_vecs.T
        for row in range(len(q)):
            np.maximum.at(nearest_ask[row], ask_owner, sims[row])
        near = (1 - w) * near + w * nearest_ask
    return near + stand

def first_right(q, doc_vecs, ask_vecs, rights, w, stand, chunk=256):
    """Per question, the rank (1 = first) of its first right survey question, None if not in the top 50."""
    out = []
    for s in range(0, len(q), chunk):
        top = np.argsort(-fused(q[s:s + chunk], doc_vecs, ask_vecs, w, stand), axis=1)[:, :50]
        out += [next((n + 1 for n, d in enumerate(row) if d in right), None) for row, right in zip(top, rights[s:s + chunk])]
    return out

def summary(ranks):
    at_k = lambda k: round(sum(1 for r in ranks if r and r <= k) / len(ranks), 3)
    return {"top1": at_k(1), "top3": at_k(3), "top10": at_k(10), "top14": at_k(14)}

print({t: len(rows) for t, rows in SETS.items()})


In [ ]:
# 3. Today's bge-small, plain and with its instruction, scored as Ask ranks (v8's yardstick). The asks' weight is chosen
# on the tuning questions; the standing is scaled to the model's own spread of closeness (the same scale maps its
# thresholds at export). v8 read the pooling from the model and got nothing (a newer sentence-transformers), so its
# export refused every model: the pooling is now the model card's.
from sentence_transformers import SentenceTransformer
BGE_QUERY = "Represent this sentence for searching relevant passages: "
EMBEDDERS = [   # bge pools the first token (its model card): set here, and checked at export against PyTorch's vectors
    {"name": "BAAI/bge-small-en-v1.5", "tag": "bge-small", "pooling": "cls"},                                  # today's
    {"name": "BAAI/bge-small-en-v1.5", "tag": "bge-small-instruction", "query": BGE_QUERY, "pooling": "cls"},  # with its instruction
]

def encode(model, texts, prefix, batch=128):
    return model.encode([prefix + t for t in texts], batch_size=batch, normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=False).astype(np.float32)

def spread(near):   # the typical spread of a question's 50 closest survey questions: the scale thresholds move with
    return float(np.mean(np.sort(near, axis=1)[:, -50:].std(axis=1)))

def cpu_ms(spec, questions):   # one question at a time on one CPU thread, as the server embeds one (PyTorch, not 8-bit)
    model = SentenceTransformer(spec["name"], device="cpu", trust_remote_code=spec.get("remote", False))
    threads = torch.get_num_threads(); torch.set_num_threads(1)
    started = time.time()
    for q in questions:
        model.encode([spec.get("query", "") + q])
    torch.set_num_threads(threads)
    return round((time.time() - started) / len(questions) * 1000, 1)

BASE, kept = None, {}

def score(model, spec):
    """Every survey question, ask and test question through this model; ranked as Ask ranks; the numbers kept."""
    global BASE
    tag, started = spec["tag"], time.time()
    qp, dp = spec.get("query", ""), spec.get("doc", "")
    doc_vecs = encode(model, [d["embed"] for d in docs], dp)
    ask_vecs = encode(model, [a for _, a in owners], qp)
    q = {t: encode(model, [r["subject"] for r in rows], qp) for t, rows in SETS.items()}
    own_spread = spread(q["tune"] @ doc_vecs.T)
    scale = own_spread / BASE["spread"] if BASE else 1.0
    stand = standing * scale
    tried = {w: summary(first_right(q["tune"], doc_vecs, ask_vecs, RIGHT["tune"], w, stand)) for w in (0.0, 0.3, 0.5, 0.7)}
    w = max(tried, key=lambda w: (tried[w]["top14"], tried[w]["top1"], -abs(w - 0.3)))
    sets = {t: summary(first_right(q[t], doc_vecs, ask_vecs, RIGHT[t], w, stand)) for t in SETS}
    alone = {t: summary(first_right(q[t], doc_vecs, ask_vecs, RIGHT[t], 0.0, 0)) for t in SETS if t != "tune"}
    kept[tag] = {"top1": fused(q["tune"], doc_vecs, ask_vecs, w, stand).max(axis=1),   # for the threshold mapping
                 "nearest_ask": (q["tune"] @ ask_vecs.T).max(axis=1), "doc_vecs": doc_vecs, "ask_vecs": ask_vecs}
    if BASE is None:
        BASE = {"spread": own_spread, "tag": tag}
    results["embedders"][tag] = {
        "name": spec["name"], "params": sum(p.numel() for p in model.parameters()), "dims": int(doc_vecs.shape[1]),
        "pooling": spec["pooling"], "query_prefix": qp, "doc_prefix": dp, "ask_weight": w, "scale": round(scale, 3),
        "weights_tried": tried, "sets": sets, "vectors_alone": alone, "seconds": round(time.time() - started),
        "cpu_ms_question": cpu_ms(spec, [r["subject"] for r in SETS["golden"][:30]])}
    r = results["embedders"][tag]
    print(f"{tag:22} {r['params'] / 1e6:5.0f}M w={w} · golden top1 {sets['golden']['top1']:.0%} top14 {sets['golden']['top14']:.0%} · "
          f"fresh {sets['fresh']['top1']:.0%} / {sets['fresh']['top14']:.0%} · bank {sets['bank']['top1']:.0%} / {sets['bank']['top14']:.0%}"
          f" · tune {sets['tune']['top14']:.0%} · {r['cpu_ms_question']} ms on 1 CPU", flush=True)
    save()

for spec in EMBEDDERS:
    try:
        model = SentenceTransformer(spec["name"], device="cuda", trust_remote_code=spec.get("remote", False),
                                    model_kwargs={"torch_dtype": torch.float16})
        score(model, spec)
    except Exception as exc:   # one model that cannot load or run is noted; the others go on
        results["embedders"][spec["tag"]] = {"name": spec["name"], "error": repr(exc)[:300]}
        print(spec["tag"], "failed:", repr(exc)[:200], flush=True)
        save()
    model = None
    free()


In [ ]:
# 4. Both fine-tuned as in v8 (v8 picked these two: the best small ones on the tuning questions) on our practice
# questions: each practice question, and one analyst's ask per survey question, against its survey question with a hard
# negative (a near miss ranked 3rd-12th that is not the answer nor its same wording), one epoch. The tuning questions
# are left out of the training, so the score after is honest.
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers
specs = {s["tag"]: s for s in EMBEDDERS}
for tag in ("bge-small", "bge-small-instruction"):
    spec = specs[tag]
    try:
        model = SentenceTransformer(spec["name"], device="cuda", trust_remote_code=spec.get("remote", False))
        qp, dp = spec.get("query", ""), spec.get("doc", "")
        rows = [(p["subject"], at[p["key"]]) for p in search["practice"] if p["q"] not in tune_q and p["key"] in at]
        rows += [(random.Random(i).choice(d["asks"]), i) for i, d in enumerate(docs) if d["asks"]]
        anchors = encode(model, [t for t, _ in rows], qp)
        doc_vecs = kept[tag]["doc_vecs"]
        rng, anchor, positive, negative = random.Random(9), [], [], []
        for s in range(0, len(rows), 1024):
            top = np.argsort(-(anchors[s:s + 1024] @ doc_vecs.T), axis=1)[:, :12]
            for (text, right), row in zip(rows[s:s + 1024], top):
                same = docs[right]["text"].lower()
                near = [d for d in row[2:] if d != right and docs[d]["text"].lower() != same]
                if near:
                    anchor.append(qp + text); positive.append(dp + docs[right]["embed"]); negative.append(dp + docs[rng.choice(near)]["embed"])
        data = Dataset.from_dict({"anchor": anchor, "positive": positive, "negative": negative})
        args = SentenceTransformerTrainingArguments(output_dir=f"ft-{tag}", num_train_epochs=1, per_device_train_batch_size=64,
                                                    learning_rate=2e-5, warmup_ratio=0.1, bf16=torch.cuda.is_bf16_supported(),
                                                    fp16=not torch.cuda.is_bf16_supported(),
                                                    batch_sampler=BatchSamplers.NO_DUPLICATES, logging_steps=100,
                                                    save_strategy="no", report_to="none")
        SentenceTransformerTrainer(model=model, args=args, train_dataset=data, loss=losses.MultipleNegativesRankingLoss(model)).train()
        model.save(f"ft-{tag}/final")
        tuned = {**spec, "name": f"ft-{tag}/final", "tag": f"{tag}-tuned", "base": spec["name"], "remote": spec.get("remote", False)}
        specs[tuned["tag"]] = tuned
        print(f"{tag}: trained on {len(anchor)} pairs", flush=True)
        score(model.half(), tuned)
        results["embedders"][tuned["tag"]]["base"] = spec["name"]
    except Exception as exc:
        results["embedders"][f"{tag}-tuned"] = {"name": spec["name"], "error": repr(exc)[:300]}
        print(tag, "fine-tune failed:", repr(exc)[:200], flush=True)
    save()
    model = None
    free()


In [ ]:
# 5. Both fine-tuned search models exported, as
# api/ask_model.py loads one: 32-bit ONNX of the transformer (its last hidden states: embed pools them), the tokenizer, and
# calibration.json with how to read it and today's thresholds mapped to its scale: an absolute one (nothing close, a
# verified match) by the same share of tuning questions on each side, an added one (standing, area, gate) by its spread.
# Checked: the file, read as the server reads it, gives PyTorch's vectors; timed on one CPU thread. 32-bit, not 8-bit:
# v8's 8-bit copies gave 0.966-0.969 of PyTorch's vectors, and their test inside Ask was against today's 16-bit file.
import onnxruntime, tokenizers
today = search["bge_small"]   # bge-small's thresholds, as Ask uses them

class Hidden(torch.nn.Module):   # bge is BERT: its last hidden states, which api/ask_model.py embed pools (cls)
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, input_ids, attention_mask, token_type_ids):
        return self.inner(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).last_hidden_state


def server_embed(folder, cal, texts, threads=0):   # api/ask_model.py embed, line for line
    words = tokenizers.Tokenizer.from_file(f"{folder}/tokenizer.json")
    words.enable_truncation(cal["length"]); words.enable_padding(pad_id=words.token_to_id(cal["pad_token"]) or 0, pad_token=cal["pad_token"])
    options = onnxruntime.SessionOptions()
    if threads:
        options.intra_op_num_threads = options.inter_op_num_threads = threads
    session = onnxruntime.InferenceSession(f"{folder}/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
    inputs = {i.name for i in session.get_inputs()}
    out = []
    for t in texts:
        c = words.encode(t)
        ids, mask = np.array([c.ids], np.int64), np.array([c.attention_mask], np.int64)
        hidden = session.run(None, {k: v for k, v in {"input_ids": ids, "attention_mask": mask, "token_type_ids": np.zeros_like(ids)}.items() if k in inputs})[0]
        v = hidden[:, 0] if cal["pooling"] == "cls" else hidden[np.arange(1), mask.sum(1) - 1] if cal["pooling"] == "last" else (hidden * mask[..., None]).sum(1) / mask.sum(1, keepdims=True)
        out.append(v[0] / np.linalg.norm(v[0]))
    return np.array(out, np.float32)

def mapped(value, base, own):   # the same share of tuning questions below the threshold, on the model's own scale
    return round(float(np.quantile(own, float(np.mean(base < value)))), 4)

for tag in ("bge-small-tuned", "bge-small-instruction-tuned"):
    spec, r = specs[tag], results["embedders"].get(tag, {})
    try:
        if "error" in r or not r:
            raise ValueError(f"not fine-tuned: {r.get('error', 'missing')}")
        if r["pooling"] not in ("cls", "mean", "last"):
            raise ValueError(f"pooling {r['pooling']} is not one the server reads")
        model = SentenceTransformer(spec["name"], device="cpu", model_kwargs={"attn_implementation": "eager"})   # traces the same at any length
        folder = f"ask-v8b/embedders/{tag}"
        os.makedirs(folder, exist_ok=True)
        tok = model.tokenizer
        dummy = tok(["What share of shoppers bought from Amazon?"], return_tensors="pt")
        names = [n for n in ("input_ids", "attention_mask", "token_type_ids") if n in dummy]
        torch.onnx.export(Hidden(model[0].auto_model).eval(), tuple(dummy[n] for n in names), f"{folder}/model_optimized.onnx",
                          input_names=names, output_names=["last_hidden_state"], opset_version=17, dynamo=False,
                          dynamic_axes={n: {0: "batch", 1: "tokens"} for n in names} | {"last_hidden_state": {0: "batch", 1: "tokens"}})
        tok.save_pretrained("tok_tmp"); shutil.copy("tok_tmp/tokenizer.json", f"{folder}/tokenizer.json")
        base, own = kept[BASE["tag"]], kept[tag]
        cal = {"pooling": r["pooling"], "query_prefix": r["query_prefix"], "doc_prefix": r["doc_prefix"], "length": 512,
               "pad_token": tok.pad_token or "[PAD]", "ask_weight": r["ask_weight"],
               "near_floor": mapped(today["near_floor"], base["top1"], own["top1"]),
               "verified_near": mapped(today["verified_near"], base["nearest_ask"], own["nearest_ask"]),
               "gate": round(today["gate"] * r["scale"], 4), "standing": round(today["standing"] * r["scale"], 4),
               "area_boost": round(today["area_boost"] * r["scale"], 4),
               "made": f"scripts/colab/train_search_model_v8b.ipynb: {r.get('base', spec['name'])}{', fine-tuned on our practice questions' if 'base' in r else ''}, 32-bit"}
        json.dump(cal, open(f"{folder}/calibration.json", "w"), indent=1)
        sample = [r_["subject"] for r_ in SETS["tune"][:200]]
        torch_vecs = encode(model, sample, cal["query_prefix"])
        onnx_vecs = server_embed(folder, cal, [cal["query_prefix"] + s for s in sample])
        started = time.time()
        server_embed(folder, cal, [cal["query_prefix"] + s for s in sample[:40]], threads=1)
        r["export"] = {"folder": folder, "calibration": cal, "same_as_pytorch": round(float(np.mean(np.sum(torch_vecs * onnx_vecs, axis=1))), 4),
                       "ms_question_1cpu": round((time.time() - started) / 40 * 1000, 1),
                       "mb": round(os.path.getsize(f"{folder}/model_optimized.onnx") / 1e6)}
        if r["export"]["same_as_pytorch"] < 0.999:   # 32-bit is PyTorch's own arithmetic (~1.0); a wrong pooling gives far less
            raise ValueError(f"the file's vectors differ from PyTorch's: {r['export']['same_as_pytorch']}")
        print(tag, json.dumps(r["export"], indent=1), flush=True)
    except Exception as exc:
        r["export"] = {"error": repr(exc)[:300]}
        print(tag, "export failed:", repr(exc)[:200], flush=True)
    save()
    model = None
    free()


In [ ]:
# 6. The zip (results.json; each exported search model a folder Ask loads with CIP_ASK_EMBED_DIR), the numbers in short
import zipfile
save()
exported = [t for t, r in results["embedders"].items() if "export" in r and "error" not in r["export"]]
assert exported, "No model exported: see the errors above and in ask-v8b/results.json (send that file)"
with zipfile.ZipFile("ask-v8b.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, names in os.walk("ask-v8b"):
        for n in names:
            z.write(os.path.join(root, n))
print(f"ask-v8b.zip: {os.path.getsize('ask-v8b.zip') / 1e6:.0f} MB\n")
for tag, r in results["embedders"].items():
    if "error" in r:
        print(f"  {tag:28} failed: {r['error'][:80]}")
        continue
    s = r["sets"]
    print(f"  {tag:28} golden {s['golden']['top1']:.0%}/{s['golden']['top14']:.0%} · fresh {s['fresh']['top1']:.0%}/{s['fresh']['top14']:.0%}"
          f" · bank {s['bank']['top1']:.0%}/{s['bank']['top14']:.0%}" + (f" · exported, vectors as PyTorch's {r['export']['same_as_pytorch']}"
          if "export" in r and "error" not in r["export"] else f" · export: {r.get('export', {}).get('error', '-')[:80]}"))
from google.colab import files
files.download("ask-v8b.zip")
print("Done: send ask-v8b.zip. Runtime → Disconnect and delete runtime stops the units.")
